# PCA（CIFAR-10 + VGG16 fc7）

このノートブックでは、CIFAR-10の画像100枚からVGG16のfc7特徴（4096次元）を抽出し、PCAで次元圧縮（寄与率95%, 90% と 128次元）を行います。その後、k-means（k=5,10）でクラスタリングし、結果を比較します。

In [1]:
# 必要なライブラリ
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import VGG16_Weights
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# 日本語フォント設定（利用可能なものを自動選択）
def set_japanese_font():
    candidates = [
        "IPAexGothic",
        "IPAGothic",
        "Noto Sans CJK JP",
        "Noto Sans JP",
        "TakaoGothic",
        "Yu Gothic",
        "Hiragino Sans",
    ]
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in candidates:
        if name in available:
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False

set_japanese_font()

# 再現性
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# デバイス
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


## 1. データ準備（CIFAR-10の100枚）

1.AutoEncoderでダウンロード済みのCIFAR-10を使用します。VGG16に合わせて224×224にリサイズし、ImageNetの平均・分散で正規化します。

In [2]:
# 画像前処理（VGG16用）
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=VGG16_Weights.IMAGENET1K_V1.transforms().mean,
                         std=VGG16_Weights.IMAGENET1K_V1.transforms().std),
])

# CIFAR-10の読み込み（1.AutoEncoderのデータを利用）
data_root = "../1.AutoEncoder/data"
train_dataset = datasets.CIFAR10(root=data_root, train=True, download=False, transform=transform)

# ランダムに100枚抽出
num_samples = 100
indices = np.random.choice(len(train_dataset), num_samples, replace=False)
subset = Subset(train_dataset, indices)

batch_size = 25
loader = DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=2)

print("samples:", len(subset))

samples: 100


## 2. VGG16のfc7特徴（4096次元）の抽出

VGG16の最後の分類層（fc8）を除いた部分を使い、fc7（4096次元）の特徴量を取得します。

In [3]:
# VGG16の読み込み（事前学習済み）
weights = VGG16_Weights.IMAGENET1K_V1
vgg = models.vgg16(weights=weights).to(device)

# fc7までを取り出す
feature_extractor = nn.Sequential(
    vgg.features,
    vgg.avgpool,
    nn.Flatten(),
    *list(vgg.classifier.children())[:-1]
).to(device)
feature_extractor.eval()

# 特徴量の抽出
features_list = []
images_list = []

with torch.no_grad():
    for imgs, _ in loader:
        imgs = imgs.to(device)
        feats = feature_extractor(imgs)
        features_list.append(feats.cpu().numpy())
        images_list.append(imgs.cpu())

features = np.concatenate(features_list, axis=0)
images = torch.cat(images_list, dim=0)
print("features shape:", features.shape)

features shape: (100, 4096)


## 3. PCAによる次元圧縮（95%, 90%, 128次元）

4096次元の特徴量をPCAで圧縮します。寄与率95%, 90%の次元数と、固定128次元を比較します。

In [ ]:
# 寄与率95%
pca_95 = PCA(n_components=0.95, random_state=seed)
feat_95 = pca_95.fit_transform(features)

# 寄与率90%
pca_90 = PCA(n_components=0.90, random_state=seed)
feat_90 = pca_90.fit_transform(features)

# 128次元
pca_128 = PCA(n_components=128, random_state=seed)
n_comp_128 = min(128, features.shape[0], features.shape[1])
if n_comp_128 != 128:
    print(f"n_components reduced to {n_comp_128} (<= min(n_samples, n_features))")
pca_128 = PCA(n_components=n_comp_128, random_state=seed)
feat_128 = pca_128.fit_transform(features)

print("dim 4096:", features.shape[1])
print("dim 95%:", feat_95.shape[1])
print("dim 90%:", feat_90.shape[1])
print("dim 128:", feat_128.shape[1])

ValueError: n_components=128 must be between 0 and min(n_samples, n_features)=100 with svd_solver='full'

## 4. k-meansクラスタリング（k=5,10）

4096次元と各PCA次元でクラスタリングし、分布を確認します。

In [ ]:
feature_sets = {
    "4096": features,
    "PCA95": feat_95,
    "PCA90": feat_90,
    "PCA128": feat_128,
}

cluster_labels = {}

for name, feat in feature_sets.items():
    for k in [5, 10]:
        kmeans = KMeans(n_clusters=k, random_state=seed, n_init=10)
        labels = kmeans.fit_predict(feat)
        cluster_labels[(name, k)] = labels
        unique, counts = np.unique(labels, return_counts=True)
        print(f"{name} k={k} counts:", dict(zip(unique, counts)))

## 5. クラスタ代表画像の可視化

各クラスタから数枚ずつ表示し、分かりやすく比較します。

In [ ]:
# 逆正規化
mean = torch.tensor(weights.transforms().mean).view(1, 3, 1, 1)
std = torch.tensor(weights.transforms().std).view(1, 3, 1, 1)

def denorm(x):
    return (x * std + mean).clamp(0, 1)

images_denorm = denorm(images)

# 表示関数
def show_cluster_samples(title, labels, n_cols=5):
    k = len(np.unique(labels))
    n_rows = k
    plt.figure(figsize=(n_cols * 2.2, n_rows * 2.2))
    plt.suptitle(title, y=1.02)

    for cluster_id in range(k):
        idxs = np.where(labels == cluster_id)[0][:n_cols]
        for j, idx in enumerate(idxs):
            ax = plt.subplot(n_rows, n_cols, cluster_id * n_cols + j + 1)
            img = images_denorm[idx].numpy()
            plt.imshow(np.transpose(img, (1, 2, 0)))
            plt.axis("off")
            if j == 0:
                ax.set_title(f"cluster {cluster_id}")

    plt.tight_layout()
    plt.show()

# 代表例の表示
for name in ["4096", "PCA95", "PCA90", "PCA128"]:
    for k in [5, 10]:
        labels = cluster_labels[(name, k)]
        show_cluster_samples(f"{name}  k={k}", labels)